# 보조 원판 모델로 복구한 장면과 영상 완성 시간

Mac MPS에서 원판 단독과 원판 두 모델 조합을 비교한다. 두 조건 모두 같은 자세 모델, 입력 640, 궤적 예외 규칙, 원본 해상도와 H.264 인코더를 사용한다. 한 영상은 한 번 읽고, 보조 모델은 별도 이력이 필요해 모든 프레임을 추론한다.

In [ ]:
from pathlib import Path
import sys
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "wlcv/config.py").is_file())
sys.path.insert(0, str(root))
import pandas as pd
from matplotlib import pyplot as plt
from wlcv.evaluation.plate_fallback_timing import OUTPUT, benchmark, verify_runs

repeat_measurement = False
if repeat_measurement:
    session = benchmark()
    verify_runs(session)
else:
    session = sorted(p.parent for p in OUTPUT.glob("*/checks.csv"))[-1]

times = pd.read_csv(session/"times.csv")
checks = pd.read_csv(session/"checks.csv")
paired = pd.read_csv(session/"paired_quality.csv")
display(times.groupby(["video","mode"]).overlay_s.agg(["count","median","min","max"]).round(3))

## 같은 영상에서 추가로 걸린 시간

각 조건·영상당 4회씩 새 프로세스에서 실행했다. 타이머는 가중치 파일을 연 뒤, 영상 열기 직전부터 마지막 프레임의 인코딩·저장이 끝날 때까지다. 첫 모델 호출 준비도 포함된다. 네 번의 개별 측정값을 같은 블록끼리 연결한다.

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(8,3.8),layout="constrained",sharey=True)
for ax, (video, group) in zip(axes, times.groupby("video")):
    wide = group.pivot(index="block",columns="mode",values="overlay_s")
    for _, row in wide.iterrows():
        ax.plot([0,1],row[["single","dual"]],marker="o",alpha=.7)
    ax.set(xticks=[0,1],xticklabels=["Single","Dual"],title=video[:3],ylabel="Seconds to saved overlay")
    ax.set_xlim(-.2,1.2)
fig.savefig(session/"overlay_times.png",dpi=150)
plt.show()
display(times[["video","block","mode","overlay_s","with_model_load_s","command_s"]].round(3))

## 추가로 복구하거나 잃은 프레임

원본을 전부 처리한 후 동작 중 수동 박스가 있는 프레임만 IoU 0.50으로 채점한다. 전체 영상에서 박스를 출력한 장수와 수동 정답에 맞은 장수를 구분한다.

In [ ]:
display(checks[["video","block","mode","decoded","scored","hits","missing_outputs","aux_selected"]])
for video, group in paired.groupby("video"):
    cases = group[group.block.eq(1)]
    print(video, "복구", cases.loc[~cases.single & cases.dual,"frame"].tolist(),
          "손실", cases.loc[cases.single & ~cases.dual,"frame"].tolist())

## 실제로 저장된 오버레이에서 확인

a03 14번과 c04 27번을 같은 블록 안에서 비교한다. 산출물은 실행별 폴더에 보존된다.

In [ ]:
import cv2

fig, axes = plt.subplots(2,2,figsize=(7,7),layout="constrained")
for row, (video, frame) in enumerate([("a03_sn_f_01",14),("c04_sn_f_01",27)]):
    for col,mode in enumerate(["single","dual"]):
        record = times[(times.video == video) & (times["mode"] == mode) & (times.block == 1)].iloc[0]
        path = session/record.folder/"overlay.mp4"
        cap = cv2.VideoCapture(str(path))
        try:
            cap.set(cv2.CAP_PROP_POS_FRAMES,frame)
            ok, image = cap.read()
            if not ok:
                raise OSError(path)
        finally:
            cap.release()
        ax = axes[row,col]
        ax.imshow(cv2.cvtColor(image,cv2.COLOR_BGR2RGB))
        ax.set_title(f"{video[:3]} f{frame} | {mode}")
        ax.axis("off")
fig.savefig(session/"saved_frames.png",dpi=145)
plt.show()